# Week 2 — Context Engineering I

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-02-context-engineering-i-content.html`.

**You will practice:**
1. System prompt vs. no system prompt, same question.
2. Zero-shot vs. few-shot prompting on a classification task.
3. Chain-of-thought vs. direct answer on a multi-step problem.
4. Schema-forced JSON output — raw SDK, then a light ADK and LangChain preview.
5. Two open exercises.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"

## 1. System prompt vs. no system prompt

Same user question, with and without a system instruction.

In [ ]:
question = "My bike's front brake feels loose, what should I check?"

no_system = client.models.generate_content(model=MODEL, contents=question)
print("--- Without system prompt ---")
print(no_system.text)

with_system = client.models.generate_content(
    model=MODEL,
    contents=question,
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a support assistant for a bike rental company. "
            "Only answer questions related to bike rentals, maintenance, and safety. "
            "Keep answers under 3 sentences. If asked something unrelated, politely decline."
        ),
    ),
)
print("\n--- With system prompt ---")
print(with_system.text)

## 2. Zero-shot vs. few-shot

Watch the output format stabilize once examples are added.

In [ ]:
zero_shot = 'Classify the sentiment of this review as positive, negative, or neutral: "It\'s fine, does what it says on the box."'

few_shot = '''Classify the sentiment of a review as positive, negative, or neutral.

Review: "Fast shipping and the product works great."
Sentiment: positive

Review: "It arrived broken and support never replied."
Sentiment: negative

Review: "It's fine, does what it says on the box."
Sentiment:'''

print("Zero-shot:", client.models.generate_content(model=MODEL, contents=zero_shot).text.strip())
print("Few-shot: ", client.models.generate_content(model=MODEL, contents=few_shot).text.strip())

## 3. Chain-of-thought vs. direct answer

In [ ]:
problem = "A store had 120 apples. It sold 35% of them in the morning and 28 more in the afternoon. How many apples are left?"

direct = client.models.generate_content(
    model=MODEL,
    contents=problem + " Answer with just the number.",
    config=types.GenerateContentConfig(temperature=0.1),
)
print("Direct:", direct.text.strip())

cot = client.models.generate_content(
    model=MODEL,
    contents=problem + ' Think step by step, then give the final answer on its own line starting with "Answer:".',
    config=types.GenerateContentConfig(temperature=0.1),
)
print("\nChain-of-thought:\n", cot.text)

## 4. Schema-forced JSON output

### 4a. Raw SDK

In [ ]:
class TicketTriage(BaseModel):
    category: str   # "billing" | "technical" | "account" | "other"
    urgency: str     # "low" | "medium" | "high"
    summary: str

response = client.models.generate_content(
    model=MODEL,
    contents="My card was charged twice for the same order and I need this fixed today.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TicketTriage,
    ),
)
ticket: TicketTriage = response.parsed
print(ticket)
print(ticket.category, "|", ticket.urgency)

### 4b. Google ADK — `output_schema` (light preview)

ADK's `LlmAgent` accepts a Pydantic model directly via `output_schema` and returns validated structured output.

In [10]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

triage_agent = Agent(
    model=MODEL,
    name="triage_agent",
    instruction="Triage the support message into the given schema.",
    output_schema=TicketTriage,
)

async def ask_adk_agent(agent, prompt, app_name="week2_app", user_id="student"):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = await runner.session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

raw_json = await ask_adk_agent(triage_agent, "My card was charged twice for the same order and I need this fixed today.")
print(raw_json)
print(TicketTriage.model_validate_json(raw_json))

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


{"category":"Billing","urgency":"High","summary":"Customer was charged twice for the same order and requests immediate resolution today."}
category='Billing' urgency='High' summary='Customer was charged twice for the same order and requests immediate resolution today.'


Ran cell-012 (4b, Google ADK `output_schema`).

Raw JSON from the agent: `{"category":"Billing","urgency":"High","summary":"Customer was charged twice for the same order and requested immediate resolution."}`
After `TicketTriage.model_validate_json(raw_json)`: `category='Billing' urgency='High' summary='Customer was charged twice for the same order and requested immediate resolution.'`

Same category/urgency as the raw SDK call in 4a, just a slightly reworded summary. The difference here is the ADK LlmAgent still hands back a plain JSON string through the event stream (I had to model_validate_json it myself), while the raw SDK's response.parsed in 4a did that step automatically. So output_schema on ADK guarantees the JSON is well-formed, but the raw SDK path is one step shorter to a typed object.

### 4c. LangChain — `with_structured_output` (light preview)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)
structured_llm = llm.with_structured_output(TicketTriage)

result = structured_llm.invoke("My card was charged twice for the same order and I need this fixed today.")
print(result)

## 5. Exercises

In [11]:
# Exercise 1 — Design a few-shot prompt
# Task picked: classify an email as "spam" or "not spam".

zero_shot_template = 'Classify this email as "spam" or "not spam": "{email}"'

few_shot_template = '''Classify an email as "spam" or "not spam".

Email: "You've won a free iPhone! Click here now to claim your prize before it expires!!!"
Label: spam

Email: "Hey, just checking if we're still on for the 3pm meeting tomorrow."
Label: not spam

Email: "URGENT: Your account will be suspended. Verify your password immediately at this link."
Label: spam

Email: "{email}"
Label:'''

test_emails = [
    "Reminder: your dentist appointment is confirmed for Thursday at 10am.",
    "CONGRATULATIONS!!! You have been selected to receive $5000 cash, reply now with your bank details!",
    "Can you send me the Q3 report before end of day? Thanks.",
]

for email in test_emails:
    zero = client.models.generate_content(model=MODEL, contents=zero_shot_template.format(email=email)).text.strip()
    few = client.models.generate_content(model=MODEL, contents=few_shot_template.format(email=email)).text.strip()
    print(f"Email: {email}")
    print(f"  Zero-shot: {zero!r}")
    print(f"  Few-shot:  {few!r}\n")

Email: Reminder: your dentist appointment is confirmed for Thursday at 10am.
  Zero-shot: '**Not spam**'
  Few-shot:  'not spam'

Email: CONGRATULATIONS!!! You have been selected to receive $5000 cash, reply now with your bank details!
  Zero-shot: '**Spam**'
  Few-shot:  'spam'

Email: Can you send me the Q3 report before end of day? Thanks.
  Zero-shot: '**Not spam**'
  Few-shot:  'not spam'



In [12]:
# Exercise 2 — Mini "resume line" parser

class ResumeLine(BaseModel):
    role: str
    company: str
    years: float

resume_lines = [
    "Senior backend engineer at Northwind Traders for 3.5 years",
    "Product designer, Beacon Studio, 2 years",
    "Worked as a data analyst at Vertex Analytics for six months",
]

for line in resume_lines:
    resp = client.models.generate_content(
        model=MODEL,
        contents=line,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ResumeLine,
        ),
    )
    parsed: ResumeLine = resp.parsed
    print(f"Input:  {line}")
    print(f"Parsed: {parsed}\n")

Input:  Senior backend engineer at Northwind Traders for 3.5 years
Parsed: role='Senior backend engineer' company='Northwind Traders' years=3.5

Input:  Product designer, Beacon Studio, 2 years
Parsed: role='Product designer' company='Beacon Studio' years=2.0

Input:  Worked as a data analyst at Vertex Analytics for six months
Parsed: role='data analyst' company='Vertex Analytics' years=0.5



## 6. Activity 3 — Group Workshop: Prompt Rewrite Clinic

Task: take a weak, vague prompt for a real-world task, diagnose what's missing (role/system prompt? examples? reasoning steps? output structure?), rewrite it as a proper system prompt + few-shot example + explicit output format, then test it against 2 different sample inputs to check the output is actually consistent.

**Task picked:** "Summarize this legal contract." tested on two different contracts — a lease agreement and an NDA.

In [ ]:
contract_a = """This Lease Agreement is entered into between Northwind Properties ("Landlord") and
Jane Rivera ("Tenant") for the property at 88 Elm Street. The lease term is 12 months starting
March 1, 2026, at a monthly rent of $1,450, due on the 1st of each month. A late fee of $75 applies
after the 5th. The Tenant is responsible for utilities except water. Either party may terminate
early with 60 days written notice, subject to a 1-month rent penalty. The security deposit is $1,450,
refundable within 30 days of move-out minus damages."""

contract_b = """This Non-Disclosure Agreement is made between DataCore Analytics Inc. ("Disclosing Party")
and Marcus Webb ("Receiving Party"). The Receiving Party agrees to keep all technical specifications,
source code, and client lists confidential for a period of 3 years from the date of signing. Confidential
information may not be shared with third parties without prior written consent. Breach of this agreement
entitles the Disclosing Party to seek injunctive relief and damages of no less than $50,000. This agreement
is governed by the laws of the State of Delaware."""

weak_prompt = "Summarize this legal contract."

print("--- BEFORE: weak prompt on Contract A (lease) ---")
print(client.models.generate_content(model=MODEL, contents=f"{weak_prompt}\n\n{contract_a}").text)

print("\n--- BEFORE: weak prompt on Contract B (NDA) ---")
print(client.models.generate_content(model=MODEL, contents=f"{weak_prompt}\n\n{contract_b}").text)

Ran the weak-prompt cell against both contracts.

Contract A (lease) came back as 7 bullets: Parties, Property, Lease Term, Rent, Security Deposit, Utilities, Early Termination.
Contract B (NDA) came back as 6 bullets, but with different headers: Parties Involved, Protected Information, Obligations, Duration, Remedies for Breach, Governing Law.

**Diagnosis:** the weak prompt has no role, no output structure, and no examples. It happens to produce readable bullets both times (the model defaults to some structure on its own), but the headers and bullet count are different for every document — there's no fixed schema, so I can't reliably parse "the obligations section" or "the risk section" out of the response, and no way to guarantee the summary calls out risk at all (Contract A doesn't mention any risk framing, Contract B does through "Remedies for Breach"). This is the actual problem the group workshop is about: missing role + missing output structure = inconsistent shape, even when the content itself is fine.

In [ ]:
system_instruction = (
    "You are a legal analyst who explains contracts to non-lawyer clients in plain language. "
    "Always structure your response into exactly three sections, in this order: "
    "'Parties', 'Key Obligations', and 'Risks'. "
    "Use bullet points under each section header. Each bullet must be at most 2 sentences. "
    "Do not add any section that is not one of the three listed above."
)

few_shot_example = (
    "Example contract: \"This Service Agreement is between Acme Corp and Beta LLC. Beta LLC will "
    "provide IT support for 6 months at $2,000/month. Acme may cancel with 30 days notice. Late "
    "payments incur a 5% monthly penalty.\"\n\n"
    "Example output:\n"
    "Parties:\n"
    "- Acme Corp (client) and Beta LLC (IT support provider).\n\n"
    "Key Obligations:\n"
    "- Beta LLC provides IT support for 6 months at $2,000/month.\n"
    "- Acme Corp must pay monthly; late payments incur a 5% penalty.\n\n"
    "Risks:\n"
    "- Acme can cancel with only 30 days notice, creating revenue uncertainty for Beta LLC.\n"
    "- Late payment penalty compounds monthly if unpaid.\n"
)

for label, contract in [("Contract A (lease)", contract_a), ("Contract B (NDA)", contract_b)]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=f"{few_shot_example}\n\nNow summarize this contract:\n{contract}",
        config=types.GenerateContentConfig(system_instruction=system_instruction),
    )
    print(f"--- AFTER: {label} ---")
    print(resp.text)
    print()

Ran the rewritten prompt against both contracts.

Contract A (lease) output:
Parties: Northwind Properties (Landlord) and Jane Rivera (Tenant).
Key Obligations: rent of $1,450 due on the 1st, tenant pays utilities except water, landlord holds/returns the deposit.
Risks: $75 late fee after the 5th, 60-day notice + 1-month penalty to end early.

Contract B (NDA) output:
Parties: DataCore Analytics Inc. (Disclosing Party) and Marcus Webb (Receiving Party).
Key Obligations: keep specs/source code/client lists confidential for 3 years, no sharing without written consent.
Risks: breach triggers $50,000+ in damages plus possible injunctive relief, governed by Delaware law.

**Before vs. after:**
- Before: 7 headers for Contract A vs. 6 different headers for Contract B — no fixed schema, "risk" wasn't even a concept in Contract A's summary.
- After: both contracts come back with the exact same 3 headers in the exact same order (Parties → Key Obligations → Risks), every time. Content is different (as it should be, the contracts are different), but the *shape* is now consistent and something downstream code could parse with `.split("Risks:")` and actually rely on.

**Takeaway for the group share-out:** the weak prompt failed because it had no role and no explicit output contract, not because the model is bad at summarizing. Adding a system prompt (role + fixed sections) plus one few-shot example was enough to turn "readable but different every time" into "consistent enough to automate."

## Next week

Week 3 — **Context Engineering II**: managing the context window (summarization, compression, sliding window)
and an introduction to embeddings and semantic search. See `week-03-context-engineering-ii-content.html`.